# <font color = Red>**Trabalho 1 - RBNN**

<font color = Blue>**Aprendizado Supervisionado**

<font color = Green>Integrantes:

Edson Eduardo Ferreira - 23908965

Gabriel Batista Chiezo - 23028678

Victor Furumoto Puttomatti - 23007606

Yan Yoshida Luz - 23911118

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Separando as database em treino e teste
80% treino

20% teste

## Iris

In [ ]:
# Lendo os arquivo que colocamos no Github
X_iris = pd.read_csv('https://raw.githubusercontent.com/VictorFP335/databasesRBNN/refs/heads/main/iris.csv')
y_iris = pd.read_csv('https://raw.githubusercontent.com/VictorFP335/databasesRBNN/refs/heads/main/iris_class.csv')

# Separando a database em treino e teste (80% treino - 20% teste)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

# Resetando os index para o futuro
X_train_iris = X_train_iris.reset_index(drop=True)
X_test_iris  = X_test_iris.reset_index(drop=True)
y_train_iris = y_train_iris.reset_index(drop=True)
y_test_iris  = y_test_iris.reset_index(drop=True)

## Abalone

In [ ]:
# Lendo os arquivo que colocamos no Github
X_abalone = pd.read_csv('https://raw.githubusercontent.com/VictorFP335/databasesRBNN/refs/heads/main/abalone.csv')
y_abalone = pd.read_csv('https://raw.githubusercontent.com/VictorFP335/databasesRBNN/refs/heads/main/abalone_class.csv')

# Fazendo uma coluna binária para cada tipo de Sexo para evitar viés
X_abalone = pd.get_dummies(X_abalone, columns=['Sex'], dtype=float)

# Separando a database em treino e teste (80% treino - 20% teste)
X_train_abalone, X_test_abalone, y_train_abalone, y_test_abalone = train_test_split(
    X_abalone, y_abalone, test_size=0.2, random_state=42
)

# Resetando os index para o futuro
X_train_abalone = X_train_abalone.reset_index(drop=True)
X_test_abalone  = X_test_abalone.reset_index(drop=True)
y_train_abalone = y_train_abalone.reset_index(drop=True)
y_test_abalone  = y_test_abalone.reset_index(drop=True)

# Lendo Desvio Padrão Sigma

In [ ]:
sigma = 1

# Aplicando RBNN para database Iris

In [ ]:
# Criando a coluna class para guardar os inputados depois
X_test_iris['class'] = ''

# Criando um df para os imputado errados com algumas colunas a mais para ajudar na análise
errors = pd.DataFrame(columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species_predict', 'species_original', 'setosa', 'versicolor', 'virginica'])

# For para andar nas linhas da test
for i in range(len(X_test_iris)):
  linha = X_test_iris.loc[i]

  # Fazendo uma váriavel para guardar os valores momentaneamente com as mesmas colunas da treino
  x = X_train_iris.copy()

  # For para calcular a diferença^2 de cada linha da test com toda a treino
  for colum in X_train_iris:
    if colum == 'class': # Pulando a coluna class pq é catogórica
      continue
    x[colum] = (X_train_iris[colum] - linha[colum])**2 # Guardando na coluna a distância entre os dois pontos

  # Agora temos que somar as distâncias e tirar a raiz para saber o peso daquela linha
  # Calculando a Euclidiana
  x['dist_euclidiana'] = np.sqrt(x['sepal length'] + x['sepal width'] + x['petal length'] + x['petal width'])
  # Depois usando a Euclidiana na função de ponderamento para RBNN
  x['peso'] = (1/sigma*np.sqrt(2*np.pi)) * np.exp( (-1/2) * ((x['dist_euclidiana']**2)/sigma**2))

  # Juntando a database provisória com a coluna class
  x = pd.merge(x, y_train_iris, left_index=True, right_index=True)

  # Filtrando a db para calcular a soma dos pontos de cada class para inputar
  setosa_ = x[x['class'] == 'Iris-setosa']
  setosa = sum(setosa_['peso'])

  versicolor_ = x[x['class'] == 'Iris-versicolor']
  versicolor = sum(versicolor_['peso'])

  virginica_ = x[x['class'] == 'Iris-virginica']
  virginica = sum(virginica_['peso'])

  # Verificando qual teve a maior pontuação e inputando
  if setosa > versicolor and setosa > virginica:
    X_test_iris.loc[i, 'class'] = 'Iris-setosa'
  elif versicolor > setosa and versicolor > virginica:
    X_test_iris.loc[i, 'class'] = 'Iris-versicolor'
  elif virginica > setosa and virginica > versicolor:
    X_test_iris.loc[i, 'class'] = 'Iris-virginica'

  # Analisando se imputou errado e guardando na df errors
  if X_test_iris.loc[i, 'class'] != y_test_iris.loc[i, 'class']:
    errors = pd.concat([
        errors,
        pd.DataFrame([{
            'sepal_length': X_test_iris.loc[i, 'sepal length'],
            'sepal_width': X_test_iris.loc[i, 'sepal width'],
            'petal_length': X_test_iris.loc[i, 'petal length'],
            'petal_width': X_test_iris.loc[i, 'petal width'],
            'species_predict': X_test_iris.loc[i, 'class'],
            'species_original': y_test_iris.loc[i, 'class'],
            'setosa': setosa,
            'versicolor': versicolor,
            'virginica': virginica
        }])
    ], ignore_index=True)

print('As seguintes linhas foram inputadas erradas: ')
errors

As seguintes linhas foram inputadas erradas: 


/tmp/ipython-input-1829077348.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  errors = pd.concat([


,sepal_length,sepal_width,petal_length,petal_width,species_predict,species_original,setosa,versicolor,virginica
0,6.1,3.0,4.9,1.8,Iris-versicolor,Iris-virginica,0.046126,59.999081,58.017264


In [ ]:
# Erro plausível pois elas eram bem similares, provavelmente está próximo a zona de transição

# Aplicando RBNN para database Abalone

In [ ]:
# Criando a coluna class para guardar os imputados depois
X_test_abalone['Rings'] = 0
MAE = 0
r = 0
r2 = 0

# For para andar nas linhas da test
for i in range(len(X_test_abalone)):
  linha = X_test_abalone.loc[i]

  # Fazendo uma váriavel para guardar os valores momentaneamente com as mesmas colunas da treino
  x = X_train_abalone.copy()

  # For para calcular a diferença^2 de cada linha da test com toda a treino
  for colum in X_train_abalone:
    if colum == 'Rings': # Pulando a coluna Rings que é categórica
      continue
    x[colum] = (X_train_abalone[colum] - linha[colum])**2 # Guardando na coluna a distância entre os dois pontos

  # Agora temos que somar as distâncias e tirar a raiz para saber o peso daquela linha
  # Calculando a Euclidiana
  x['dist_euclidiana'] = np.sqrt(x['Length'] + x['Diameter'] + x['Height'] + x['Whole_weight'] + x['Shucked_weight'] + x['Viscera_weight'] + x['Shell_weight'] + x['Sex_F'] + x['Sex_I']	+ x['Sex_M'])
  # Depois usando a Euclidiana na função de ponderamento para RBNN
  x['peso'] = (1/sigma*np.sqrt(2*np.pi)) * np.exp( (-1/2) * ((x['dist_euclidiana']**2)/sigma**2))

  # Juntando a database provisória com a coluna Rings
  x = pd.merge(x, y_train_abalone, left_index=True, right_index=True)

  # Verificando qual imputar multiplicando o peso pelos rings
  x['ponderamento'] = x['peso'] * x['Rings']

  # Inputando a media
  input = (x['ponderamento'].sum()) / (x['peso'].sum())
  X_test_abalone.loc[i, 'Rings'] = int(input.round(0)) # Garantindo que seja um int

  # Calculando as métricas
  diff = y_test_abalone.loc[i, 'Rings'] - X_test_abalone.loc[i, 'Rings']
  diff2 = y_test_abalone.loc[i, 'Rings'] - y_train_abalone['Rings'].mean()
  MAE += abs(diff)
  r += diff**2
  r2 += diff2**2

MAE = MAE / len(X_test_abalone)
RMSE = np.sqrt(r / len(X_test_abalone))
r = 1 - (r/r2)
print(f'MAE = {MAE.round(4)}, RMSE = {RMSE.round(4)} e r² = {r.round(4)}')

MAE = 2.1148, MSE = 2.9643 e r² = 0.1885


# Variando o desvio padrão (σ)


In [ ]:
# Vamos variar o desvio padrão de acordo com a seguinte lista:
valores = [0.01, 0.1, 1, 5, 10]

## Aplicaçao na Iris

In [ ]:
for sigma in valores:
  # Criando a coluna class para guardar os inputados depois
  X_test_iris['class'] = ''

  # Criando um df para os imputado errados com algumas colunas a mais para ajudar na análise
  errors = 0

  # For para andar nas linhas da test
  for i in range(len(X_test_iris)):
    linha = X_test_iris.loc[i]

    # Fazendo uma váriavel para guardar os valores momentaneamente com as mesmas colunas da treino
    x = X_train_iris.copy()

    # For para calcular a diferença^2 de cada linha da test com toda a treino
    for colum in X_train_iris:
      if colum == 'class': # Pulando a coluna class que é categórica
        continue
      x[colum] = (X_train_iris[colum] - linha[colum])**2 # Guardando na coluna a distância entre os dois pontos

    # Agora temos que somar as distâncias e tirar a raiz para saber o peso daquela linha
    # Calculando a Euclidiana
    x['dist_euclidiana'] = np.sqrt(x['sepal length'] + x['sepal width'] + x['petal length'] + x['petal width'])
    # Depois usando a Euclidiana na função de ponderamento para RBNN
    x['peso'] = (1/sigma*np.sqrt(2*np.pi)) * np.exp( (-1/2) * ((x['dist_euclidiana']**2)/sigma**2))

    # Juntando a database provisória com a coluna class
    x = pd.merge(x, y_train_iris, left_index=True, right_index=True)

    # Filtrando a db para calcular a soma dos pontos de cada class para inputar
    setosa_ = x[x['class'] == 'Iris-setosa']
    setosa = sum(setosa_['peso'])

    versicolor_ = x[x['class'] == 'Iris-versicolor']
    versicolor = sum(versicolor_['peso'])

    virginica_ = x[x['class'] == 'Iris-virginica']
    virginica = sum(virginica_['peso'])

    # Verificando qual teve a maior pontuação e inputando
    if setosa > versicolor and setosa > virginica:
      X_test_iris.loc[i, 'class'] = 'Iris-setosa'
    elif versicolor > setosa and versicolor > virginica:
      X_test_iris.loc[i, 'class'] = 'Iris-versicolor'
    elif virginica > setosa and virginica > versicolor:
      X_test_iris.loc[i, 'class'] = 'Iris-virginica'

    # Analisando se imputou errado e guardando na df errors
    if X_test_iris.loc[i, 'class'] != y_test_iris.loc[i, 'class']:
      errors+=1

  print(f'Foram imputados {errors} linhas com sigma = {sigma}')


Foram imputados 5 linhas com sigma = 0.01
Foram imputados 0 linhas com sigma = 0.1
Foram imputados 1 linhas com sigma = 1
Foram imputados 6 linhas com sigma = 5
Foram imputados 11 linhas com sigma = 10


## Aplicação na Abalone


In [ ]:
for sigma in valores:
  # Criando a coluna class para guardar os imputados depois
  X_test_abalone['Rings'] = 0
  MAE = 0
  r = 0
  r2 = 0

  # For para andar nas linhas da test
  for i in range(len(X_test_abalone)):
    linha = X_test_abalone.loc[i]

    # Fazendo uma váriavel para guardar os valores momentaneamente com as mesmas colunas da treino
    x = X_train_abalone.copy()

    # For para calcular a diferença^2 de cada linha da test com toda a treino
    for colum in X_train_abalone:
      if colum == 'Rings': # Pulando a coluna Rings
        continue
      x[colum] = (X_train_abalone[colum] - linha[colum])**2 # Guardando na coluna a distância entre os dois pontos

    # Agora temos que somar as distâncias e tirar a raiz para saber o peso daquela linha
    # Calculando a Euclidiana
    x['dist_euclidiana'] = np.sqrt(x['Length'] + x['Diameter'] + x['Height'] + x['Whole_weight'] + x['Shucked_weight'] + x['Viscera_weight'] + x['Shell_weight'] + x['Sex_F'] + x['Sex_I']	+ x['Sex_M'])
    # Depois usando a Euclidiana na função de ponderamento para RBNN
    x['peso'] = (1/sigma*np.sqrt(2*np.pi)) * np.exp( (-1/2) * ((x['dist_euclidiana']**2)/sigma**2))

    # Juntando a database provisória com a coluna Rings
    x = pd.merge(x, y_train_abalone, left_index=True, right_index=True)

    # Verificando qual imputar multiplicando o peso pelos rings
    x['ponderamento'] = x['peso'] * x['Rings']

    # Inputando a media
    input = (x['ponderamento'].sum()) / (x['peso'].sum())
    X_test_abalone.loc[i, 'Rings'] = int(input.round(0)) # Garantindo que seja um int

    # Calculando as métricas
    diff = y_test_abalone.loc[i, 'Rings'] - X_test_abalone.loc[i, 'Rings']
    diff2 = y_test_abalone.loc[i, 'Rings'] - y_train_abalone['Rings'].mean()
    MAE += abs(diff)
    r += diff**2
    r2 += diff2**2

  MAE = MAE / len(X_test_abalone)
  RMSE = np.sqrt(r / len(X_test_abalone))
  r = 1 - (r/r2)
  print(f'Sigma = {sigma} | MAE = {MAE.round(4)}, RMSE = {RMSE.round(4)} e r² = {r.round(4)} ')

Sigma = 0.01 | MAE = 1.7344, RMSE = 2.5733 e r² = 0.3884 
Sigma = 0.1 | MAE = 1.6352, RMSE = 2.4247 e r² = 0.457 
Sigma = 1 | MAE = 2.1148, RMSE = 2.9643 e r² = 0.1885 
Sigma = 5 | MAE = 2.3852, RMSE = 3.292 e r² = -0.0008 
Sigma = 10 | MAE = 2.3852, RMSE = 3.292 e r² = -0.0008 


In [ ]:
'''
Conclusão Final
Quanto mais afastado de 0.1 sigma for, mais erros na inputação esse modelo irá cometer.

Então o sigma controla a “largura” da base radial:
- Sigma pequeno → modelo mais local e mais sensível.
- Sigma grande → modelo global e pouco sensível.

Pois quando aumentado o desvio padrão mais vizinhos serão ponderados com um peso maior,
vizinhos estes que estão mais afastados da amostra em questão, tornando a inputação mais sucetivel a erros.

Portanto o método RBNN depende fortemente do valor de sigma, devendo ser calibrado corretamente.

'''

'\nConclusão Final\nQuanto mais afastado de 0.1 sigma for, mais erros na inputação esse modelo irá cometer.\n\nEntão o sigma controla a “largura” da base radial:\n- Sigma pequeno → modelo mais local e mais sensível.\n- Sigma grande → modelo global e pouco sensível.\n\nPois quando aumentado o desvio padrão mais vizinhos serão ponderados com um peso maior,\nvizinhos estes que estão mais afastados da amostra em questão, tornando a inputação mais sucetivel a erros.\n\nPortanto o método RBNN depende fortemente do valor de sigma, devendo ser calibrado corretamente.\n\n'